# DLL-GAN on Google Colab (free T4 GPU)

Trains the DLL-GAN reimplementation end-to-end: degradation regressor -> GAN with the
learnable degradation loss -> sample outputs + PSNR/SSIM.

**Before you Run All:**
1. Set `REPO_URL` below to your GitHub repo (make it **public**).
2. **Runtime -> Change runtime type -> T4 GPU**.
3. Data: nothing to do — the notebook downloads the DIV2K validation set (100 images)
   automatically. Internet is on by default in Colab.

The cells auto-detect Colab vs Kaggle, so the same notebook runs on both. Scoped to
finish inside one free session.

In [ ]:
!rm -rf /content/dll-gan

In [ ]:
# ============ SET THESE ============
REPO_URL   = "https://github.com/abdessamed-britah/new-gll-gan.git"  # <-- your repo (public)
TASK       = "sr"      # "sr" | "denoise" | "jpeg"
GAN_LEVEL  = 4         # sr: 4/8/16 | denoise: 15/25/50/70 | jpeg: 10/20/30
IMAGE_SIZE = 256       # 128 trains fast on a T4; raise to 256 for more quality
MAX_IMAGES = 800       # subset of DIV2K used for training (speed)
REG_EPOCHS = 8
GAN_EPOCHS = 150
REG_BATCH  = 32
GAN_BATCH  = 8

# ===================================

In [ ]:
# --- detect environment and pick the working dir ---
import os, sys, subprocess
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False
IN_KAGGLE = os.path.isdir("/kaggle")
WORK = "/content" if IN_COLAB else ("/kaggle/working" if IN_KAGGLE else os.getcwd())
print("env:", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local", "| WORK =", WORK)

env: Colab | WORK = /content


In [ ]:
import torch, torchvision
print(torch.__version__, torchvision.__version__, torch.cuda.is_available())

2.11.0+cpu 0.26.0+cpu False


In [ ]:
# --- clone the repo ---
REPO_DIR = os.path.join(WORK, "dll-gan")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, REPO_DIR)
print("repo:", REPO_DIR)
print(sorted(os.listdir(REPO_DIR)))

repo: /content/dll-gan
['.git', '.gitignore', 'README.md', '__pycache__', 'checkpoints', 'config.py', 'data', 'datasets.py', 'degradations.py', 'evaluate.py', 'infer.py', 'losses.py', 'models.py', 'requirements.txt', 'smoke_test.py', 'train_gan.py', 'train_regressor.py']


In [ ]:
# --- environment / GPU check ---
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("!! No GPU. Colab: Runtime -> Change runtime type -> T4 GPU. "
          "Kaggle: Settings -> Accelerator -> GPU. Then re-run.")
device = "cuda" if torch.cuda.is_available() else "cpu"

torch 2.11.0+cpu | torchvision 0.26.0+cpu
CUDA available: False
!! No GPU. Colab: Runtime -> Change runtime type -> T4 GPU. Kaggle: Settings -> Accelerator -> GPU. Then re-run.


### Data

By default the notebook downloads the DIV2K validation set (100 images). If you'd
rather use your own images from Google Drive, run this instead of relying on the
download, then set `DATA_DIR` to your folder:

```python
from google.colab import drive; drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/div2k'   # your folder of clean images
```

In [ ]:
# --- get DIV2K images (Kaggle input if present, else download) ---
import glob, shutil, random
DATA_DIR = os.path.join(WORK, "data", "div2k")
os.makedirs(DATA_DIR, exist_ok=True)

def find_kaggle_inputs():
    files = []
    if os.path.isdir("/kaggle/input"):
        for ext in ("png", "jpg", "jpeg"):
            files += glob.glob(f"/kaggle/input/**/*.{ext}", recursive=True)
    files.sort(key=lambda f: ("div2k" not in f.lower(), "hr" not in f.lower(), f))
    return files

imgs = find_kaggle_inputs()          # empty on Colab
if not imgs:
    print("Downloading DIV2K train HR (800 images, ~3.5 GB)...")
    url = "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip"
    zip_path = f"{WORK}/div2k_train.zip"
    try:
        subprocess.run(["wget", "-q", url, "-O", zip_path], check=True)
        subprocess.run(["unzip", "-q", "-o", zip_path, "-d", WORK], check=True)
        imgs = glob.glob(f"{WORK}/DIV2K_train_HR/*.png")
    except Exception as e:
        raise SystemExit(f"Download failed ({e}). Mount Google Drive and point "
                         "DATA_DIR at a folder of images instead (see the note above).")

random.seed(0); random.shuffle(imgs); imgs = imgs[:MAX_IMAGES]
for f in os.listdir(DATA_DIR):       # clear any previous run
    os.remove(os.path.join(DATA_DIR, f))
for i, f in enumerate(imgs):         # copy a clean subset the repo datasets can read
    shutil.copy(f, os.path.join(DATA_DIR, f"{i:04d}{os.path.splitext(f)[1]}"))
print(f"using {len(imgs)} images -> {DATA_DIR}")

In [ ]:
# --- build a config from the repo, overridden for this run ---
from config import Config
from datasets import RegressorDataset, GANDataset
from models import Generator, PixelDiscriminator, Regressor
from losses import GANLoss, discriminator_loss, generator_loss

cfg = Config()
cfg.task            = TASK
cfg.image_size      = IMAGE_SIZE
cfg.clean_dir       = DATA_DIR
cfg.reg_epochs      = REG_EPOCHS
cfg.gan_epochs      = GAN_EPOCHS
cfg.reg_batch_size  = REG_BATCH
cfg.gan_batch_size  = GAN_BATCH
cfg.gan_level_value = GAN_LEVEL
cfg.reg_ckpt = os.path.join(REPO_DIR, "checkpoints", "regressor.pth")
cfg.gen_ckpt = os.path.join(REPO_DIR, "checkpoints", "generator.pth")
os.makedirs(os.path.dirname(cfg.reg_ckpt), exist_ok=True)
print("task:", cfg.task, "| all levels:", cfg.levels, "| training G on level:", cfg.gan_level_value)

## Step 1 - train the degradation regressor

In [ ]:
from torch.utils.data import DataLoader

def make_regressor():
    try:
        return Regressor(pretrained=True)      # ImageNet warm start
    except Exception as e:
        print("pretrained weights unavailable -> from scratch:", e)
        return Regressor(pretrained=False)

reg_ds = RegressorDataset(cfg.clean_dir, cfg.task, cfg.levels, cfg.image_size)
reg_dl = DataLoader(reg_ds, batch_size=cfg.reg_batch_size, shuffle=True, num_workers=2)

R = make_regressor().to(device)
opt = torch.optim.Adam(R.parameters(), lr=cfg.reg_lr)
mse = torch.nn.MSELoss()
R.train()
for epoch in range(cfg.reg_epochs):
    run = seen = 0
    for imgs_b, levels_b in reg_dl:
        imgs_b, levels_b = imgs_b.to(device), levels_b.to(device)
        loss = mse(R(imgs_b), levels_b)                 # paper Eq. 1
        opt.zero_grad(); loss.backward(); opt.step()
        run += loss.item() * imgs_b.size(0); seen += imgs_b.size(0)
    print(f"[R] epoch {epoch+1}/{cfg.reg_epochs}  mse={run/seen:.4f}")
torch.save(R.state_dict(), cfg.reg_ckpt)
print("saved ->", cfg.reg_ckpt)

## Step 2 - train DLL-GAN (frozen regressor as the third loss)

In [ ]:
gan_ds = GANDataset(cfg.clean_dir, cfg.task, cfg.gan_level_value, cfg.image_size)
gan_dl = DataLoader(gan_ds, batch_size=cfg.gan_batch_size, shuffle=True, num_workers=2)

G = Generator().to(device)
D = PixelDiscriminator().to(device)
R.eval()                                    # freeze R: grads flow through it, weights don't move
for p in R.parameters():
    p.requires_grad_(False)

gan_loss = GANLoss().to(device)
opt_g = torch.optim.Adam(G.parameters(), lr=cfg.gan_lr, betas=cfg.gan_betas)
opt_d = torch.optim.Adam(D.parameters(), lr=cfg.gan_lr, betas=cfg.gan_betas)

for epoch in range(cfg.gan_epochs):
    for deg, clean in gan_dl:
        deg, clean = deg.to(device), clean.to(device)
        fake = G(deg)
        opt_d.zero_grad()
        d_loss = discriminator_loss(D, clean, fake, gan_loss)
        d_loss.backward(); opt_d.step()
        opt_g.zero_grad()
        g_loss, parts = generator_loss(D, R, fake, clean, gan_loss,
                                       cfg.lambda_gan, cfg.lambda_l1, cfg.lambda_r)
        g_loss.backward(); opt_g.step()
    if epoch == 0 or (epoch + 1) % 5 == 0:
        print(f"[GAN] {epoch+1}/{cfg.gan_epochs}  D={d_loss.item():.3f} G={g_loss.item():.3f} "
              f"adv={parts['adv']:.3f} l1={parts['l1']:.3f} deg={parts['deg']:.3f}")
torch.save(G.state_dict(), cfg.gen_ckpt)
print("saved ->", cfg.gen_ckpt)

## Step 3 - sample outputs + PSNR/SSIM

These samples come from the training pool, so treat the numbers as a sanity check.
For figures to quote in your email, set `EVAL_DIR` to a **held-out** benchmark folder
(Set5 / Set14 / BSD100 for SR) and re-run this cell.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def to_img(t):                              # (3,H,W) in [-1,1] -> HxWx3 in [0,1]
    x = (t.detach().cpu().clamp(-1, 1) + 1) / 2
    return x.permute(1, 2, 0).numpy()

EVAL_DIR = None                             # e.g. "/content/set14" for a real benchmark
eval_ds = GANDataset(EVAL_DIR or cfg.clean_dir, cfg.task, cfg.gan_level_value, cfg.image_size)

G.eval()
n = min(4, len(eval_ds))
ps, ss = [], []
fig, ax = plt.subplots(n, 3, figsize=(9, 3 * n))
with torch.no_grad():
    for i in range(n):
        deg, clean = eval_ds[i]
        out = G(deg.unsqueeze(0).to(device))[0]
        di, oi, ci = to_img(deg), to_img(out), to_img(clean)
        ps.append(psnr(ci, oi, data_range=1.0))
        ss.append(ssim(ci, oi, channel_axis=2, data_range=1.0))
        for j, (im, ttl) in enumerate([(di, "degraded"), (oi, "DLL-GAN"), (ci, "ground truth")]):
            ax[i, j].imshow(np.clip(im, 0, 1)); ax[i, j].set_title(ttl); ax[i, j].axis("off")
plt.tight_layout(); plt.show()
print(f"mean PSNR {np.mean(ps):.2f} dB | mean SSIM {np.mean(ss):.4f}  (n={n})")

In [ ]:
# --- download the trained generator ---
print("generator checkpoint:", cfg.gen_ckpt)
if IN_COLAB:
    from google.colab import files
    files.download(cfg.gen_ckpt)            # triggers a browser download
else:
    print("Grab it from the Output panel (Kaggle) or the file browser.")

## Done

You have `generator.pth` (and `regressor.pth` in `dll-gan/checkpoints/`). Run inference
on your CPU at home.

**To improve results:** raise `IMAGE_SIZE` to 256, `MAX_IMAGES` to 800 (full DIV2K
train set), and `GAN_EPOCHS` to 150-200. Colab free sessions can run up to ~12h but
disconnect after ~90 min idle, so keep the tab active. If you get throttled, Kaggle's
fixed 30h/week quota is the reliable fallback - this same notebook runs there too.